# NB07 · The two RAG systems — and why they lose

> **How to read this notebook.** Every section starts with a plain-English explanation
> before any code. If you only read the text and skip the code, you will still understand
> what was done and why. No machine-learning background is assumed.
>
> Some cells need the full document collection, which lives on the DGX server
> (`/raid/hardik/Healthcare_thesis`). Those cells show the real output they produced, so you
> can read the results without re-running anything.

---
**RAG** = Retrieval-Augmented Generation. In plain words: *search the document for the answer,
then show what you found to an AI and ask it to answer.* It is the standard approach and it is
genuinely excellent — for questions whose answers are printed in the text.

In [ ]:
import sys, pathlib
# find healthcare-thesis/src no matter which folder this notebook is opened from
root = pathlib.Path.cwd()
for _ in range(5):
    if (root / "healthcare-thesis" / "src").exists():
        break
    root = root.parent
SRC = root / "healthcare-thesis" / "src"
sys.path.insert(0, str(SRC))
import warnings; warnings.filterwarnings("ignore")
print("source code folder:", SRC)

## 1 · Vanilla RAG — the 2023 standard  (`src/vanilla_rag.py`)

1. Chop the document into 1,000-character chunks, overlapping by 200 so no sentence is cut in half
2. Turn each chunk into 1,024 numbers ("embedding") that capture its meaning
3. Turn **one fixed question** into numbers the same way
4. Keep the 20 closest chunks, restored to page order
5. Put them in a prompt with the 73 technique names, ask Qwen-7B for a JSON row

In [ ]:
from vanilla_rag import QUERY, VanillaRag
print("the ONE question reused for every document:")
print("  ", QUERY)
rag = VanillaRag()
print()
print("chunk size 1000, overlap 200, keep top", rag.top_k, ", context capped at",
      rag.max_context_chars, "characters")

the ONE question reused for every document:
   antimicrobial resistance intervention: its name, objectives, activities, implementation and monitoring

chunk size 1000, overlap 200, keep top 20 , context capped at 5000 characters

## 2 · Modern RAG — every 2026 upgrade  (`src/sota_rag.py`)

Deliberately **not** a weak opponent. If we beat this, the comparison means something.

* **hybrid retrieval** — meaning-search *and* keyword-search, merged by reciprocal rank fusion,
  because the two fail in different ways
* **multi-query** — ten questions instead of one: the programme's identity plus one per ERIC
  family, so evidence about money is fetched separately from evidence about training
* **cross-encoder rerank** — a second model re-reads question and chunk *together* and re-orders
* **retrieved worked examples** — the two most similar already-completed rows, shown in full
* **structured output** — one JSON object per document

In [ ]:
from sota_rag import CATEGORY_QUERIES, IDENTITY_QUERY
print("identity query:", IDENTITY_QUERY)
print()
for c, q in list(CATEGORY_QUERIES.items())[:4]:
    print(f"  {c:42s} -> {q}")
print(f"  ... {len(CATEGORY_QUERIES)} family queries in total")

identity query: the name, objectives and main activities of the programme this document establishes

  Use evaluative and iterative strategies    -> activities showing use evaluative and iterative strategies
  Provide interactive assistance             -> activities showing provide interactive assistance
  Adapt and tailor to context                -> activities showing adapt and tailor to context
  Develop stakeholder interrelationships     -> activities showing develop stakeholder interrelationships
  ... 9 family queries in total

## 3 · The results

| | name F1 | classification | **techniques F1** | composite |
|---|---|---|---|---|
| Vanilla RAG | 42.9 | 71.1 | **9.2** | 41.0 |
| Modern RAG | 46.5 | 66.7 | **21.2** | 44.8 |
| Our system | 54.8 | 76.3 | — | **57.9** |

Every upgrade helped: 9.2 → 21.2 is more than double. **The engineering is not the problem.**
But 21.2 is still under half of what a plain word-counter achieves, because better searching
cannot find something that was never written down.

In [ ]:
import json
for f in ("vanilla_rag_score.json", "sota_rag_score.json"):
    d = json.load(open(SRC.parent/"outputs"/f))
    print(f"{f:24s} eric {d['eric_f1']:5.1f}  name {d['extraction_f1']:5.1f}  composite {d['composite']}")

vanilla_rag_score.json   eric   9.2  name  42.9  composite 41.04
sota_rag_score.json      eric  21.2  name  46.5  composite 44.8

> ## An honest caveat, currently open
>
> **We wrote these prompts too.** One prompt elsewhere in this project was later found to encode
> rules the human experts never used, and it cost that component everything (see **NB10**). If
> these RAG prompts carry a similar flaw, then part of our "+13.1 point lead" is our own setup
> rather than our method being better.
>
> Two known limitations already: the RAG systems were given technique **names but not
> definitions** (they did not fit in GPU memory — see NB04 Use 6), and their prompts are capped
> at 5,000 characters of retrieved context. Both are being audited before the comparison is
> defended in the thesis.